In [3]:
from bisect import bisect_left
from pathlib import Path
import re

import pandas as pd

folder_gambar = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\Gambar Final"
)
file_sensor = Path(
    r"D:\00 Kuliah\TA\Experiment\Data Kultivasi Mikroalga K1-16\gabungan\GABUNGAN_sensor_menit.csv"
)
file_hasil = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\sell mikroalga\gabungan_sensor_cell_density_with_image.csv")

# None berarti gunakan semua kolom sensor.
# Untuk memilih kolom tertentu, ganti None dengan daftar nama kolom.
kolom_sensor_pilihan = ["1_bga(cells/mL)"]
# Contoh:
# kolom_sensor_pilihan = ["1_bga(cells/mL)", "1_ph", "od680_estimasi(AU)"]

pola_timestamp = re.compile(
    r"obsbot_photo_(\d{8})_(\d{6})_",
    re.IGNORECASE,
)

if not folder_gambar.is_dir():
    raise NotADirectoryError(f"Folder gambar tidak ditemukan: {folder_gambar}")
if not file_sensor.is_file():
    raise FileNotFoundError(f"File sensor tidak ditemukan: {file_sensor}")

# Baca semua gambar JPG dan ambil timestamp dari nama file.
gambar_rows = []

for path in sorted(folder_gambar.rglob("*")):
    if not path.is_file() or path.suffix.lower() != ".jpg":
        continue

    cocok = pola_timestamp.search(path.name)
    if not cocok:
        raise ValueError(
            f"Timestamp tidak ditemukan pada nama gambar: {path.name}"
        )

    timestamp_gambar = pd.Timestamp(
        pd.to_datetime(
            f"{cocok.group(1)}_{cocok.group(2)}",
            format="%Y%m%d_%H%M%S",
        )
    )

    gambar_rows.append({
        "Nama_Gambar": path.name,
        "Path_Gambar": str(path),
        "Timestamp_Gambar": timestamp_gambar,
    })

if not gambar_rows:
    raise FileNotFoundError(f"Tidak ditemukan file JPG di: {folder_gambar}")

gambar = pd.DataFrame(gambar_rows)

# Baca semua data sensor dan parse timestamp.
sensor = pd.read_csv(file_sensor, encoding="utf-8-sig")

if "timestamp" not in sensor.columns:
    raise KeyError("Kolom 'timestamp' tidak ditemukan di file sensor.")

sensor["timestamp_dt"] = pd.to_datetime(
    sensor["timestamp"],
    errors="coerce",
)

if sensor["timestamp_dt"].isna().any():
    baris_invalid = (sensor.index[sensor["timestamp_dt"].isna()] + 2).tolist()
    raise ValueError(f"Timestamp sensor tidak valid pada baris: {baris_invalid}")

if len(sensor) < len(gambar):
    raise ValueError(
        f"Baris sensor ({len(sensor)}) lebih sedikit daripada gambar "
        f"({len(gambar)}), jadi tidak mungkin memasangkan sensor unik ke setiap gambar."
    )

# Pilih kolom sensor yang ingin dimasukkan ke hasil.
if kolom_sensor_pilihan is None:
    kolom_sensor_pilihan = [
        kolom for kolom in sensor.columns
        if kolom != "timestamp_dt"
    ]
else:
    kolom_tidak_ada = [
        kolom for kolom in kolom_sensor_pilihan
        if kolom not in sensor.columns
    ]
    if kolom_tidak_ada:
        raise KeyError(f"Kolom sensor tidak ditemukan: {kolom_tidak_ada}")

# Urutkan sensor menurut waktu. Setiap baris sensor hanya boleh dipakai sekali.
sensor_records = sorted(
    [
        (timestamp, index)
        for index, timestamp in enumerate(sensor["timestamp_dt"])
    ],
    key=lambda item: item[0],
)
waktu_sensor = [record[0] for record in sensor_records]
sensor_terpakai = set()

# Proses gambar secara kronologis dan pilih sensor terdekat yang belum dipakai.
urutan_gambar = sorted(
    range(len(gambar)),
    key=lambda i: gambar.iloc[i]["Timestamp_Gambar"],
)
pasangan_sensor = [None] * len(gambar)

for indeks_gambar in urutan_gambar:
    timestamp_gambar = gambar.iloc[indeks_gambar]["Timestamp_Gambar"]
    posisi = bisect_left(waktu_sensor, timestamp_gambar)

    kiri = posisi - 1
    kanan = posisi

    while kiri >= 0 and sensor_records[kiri][1] in sensor_terpakai:
        kiri -= 1

    while kanan < len(sensor_records) and sensor_records[kanan][1] in sensor_terpakai:
        kanan += 1

    kandidat = []
    if kiri >= 0:
        kandidat.append(kiri)
    if kanan < len(sensor_records):
        kandidat.append(kanan)

    indeks_record = min(
        kandidat,
        key=lambda i: (
            abs(waktu_sensor[i] - timestamp_gambar),
            waktu_sensor[i],
        ),
    )

    indeks_sensor = sensor_records[indeks_record][1]
    sensor_terpakai.add(indeks_sensor)
    pasangan_sensor[indeks_gambar] = indeks_sensor

# Gabungkan informasi gambar dan baris sensor yang dipilih.
sensor_hasil = (
    sensor.iloc[pasangan_sensor][kolom_sensor_pilihan]
    .reset_index(drop=True)
    .add_prefix("sensor_")
)

hasil = pd.concat(
    [
        gambar.reset_index(drop=True),
        sensor_hasil,
    ],
    axis=1,
)

hasil["Selisih_Dari_Sensor"] = [
    str(
        abs(
            gambar.iloc[i]["Timestamp_Gambar"]
            - sensor.iloc[pasangan_sensor[i]]["timestamp_dt"]
        )
    )
    for i in range(len(gambar))
]

hasil.to_csv(file_hasil, index=False, encoding="utf-8-sig")

print(f"Jumlah gambar: {len(gambar)}")
print(f"Gambar yang mendapat sensor unik: {len(sensor_terpakai)}")
print(f"Jumlah baris hasil: {len(hasil)}")
print(f"Selisih waktu terbesar: {max(hasil['Selisih_Dari_Sensor'])}")
print(f"CSV hasil: {file_hasil}")

Jumlah gambar: 253
Gambar yang mendapat sensor unik: 253
Jumlah baris hasil: 253
Selisih waktu terbesar: 9 days 16:42:31
CSV hasil: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\sell mikroalga\gabungan_sensor_cell_density_with_image.csv
